# Cooperative Coevolution

In cooperative coevolution, individuals do not compete — they collaborate.

Instead of two opposing populations:
- We evolve multiple subcomponents of a solution
- Fitness depends on how well components work together

Key idea:
Individuals are evaluated as part of a team, not individually.

## Core Idea

- A solution is composed of multiple parts
- Each population evolves one part of the solution
- Fitness is based on the combined performance of these parts

This introduces:
- Credit assignment problem
- Interdependence between components

## Problem

We want to approximate a function:

$f(x) = 0.5 x^2 - 0.3 x$

We will evolve a solution composed of two subcomponents:
- Component A
- Component B

Each population evolves parameters for one component.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt

def target_function(x):
    return 0.5 * x**2 - 0.3 * x 

## Representation

We split the solution into two populations:

- Population A → controls parameters (a)
- Population B → controls parameters (b)

Each individual contributes part of the final function.

Final solution:
    f(x) = component_A(x) + component_B(x)

In [ ]:
pop_size = 30
generations = 50

def init_population():
    return np.random.uniform(-1, 1, (pop_size, 2))  # each individual has 2 parameters

## Fitness Evaluation

Each individual is evaluated in combination with individuals from the other population.

We compute fitness by:
- Pairing individuals from both populations
- Evaluating the combined function
- Measuring error against the target function

In [ ]:
def component_a(x, params):
    return params[0] * x * x

def component_b(x, params):
    return params[0] * x

def evaluate_pair(a, b, x_samples):
    preds = component_a(x_samples, a) + component_b(x_samples, b)
    return np.mean((preds - target_function(x_samples))**2)

def evaluate_population(popA, popB):
    fitnessA = np.zeros(len(popA))
    fitnessB = np.zeros(len(popB))

    x_samples = np.linspace(-5, 5, 50)

    for i, a in enumerate(popA):
        for b in popB:
            error = evaluate_pair(a, b, x_samples)
            fitnessA[i] += -error  # lower error is better
            fitnessB[np.where(popB == b)[0][0]] += -error

    return fitnessA, fitnessB

## Evolution Process

We evolve each population independently:

1. Evaluate individuals based on cooperation
2. Select the best individuals
3. Reproduce with mutation
4. Repeat

Important:
Fitness depends on partners from the other population

In [ ]:
def select(pop, fitness):
    idx = np.argsort(fitness)[-len(pop)//2:]
    return pop[idx]

def reproduce(pop):
    children = []
    while len(children) < len(pop):
        parent = random.choice(pop)
        child = parent + np.random.normal(0, 0.1, size=2)
        children.append(child)
    return np.array(children)

In [ ]:
best_solutions = []

popA = init_population()
popB = init_population()

for gen in range(generations):
    fitA, fitB = evaluate_population(popA, popB)

    # Store best individuals
    best_a = popA[np.argmax(fitA)]
    best_b = popB[np.argmax(fitB)]
    best_solutions.append((gen, best_a, best_b))

    # Evolve
    popA = reproduce(select(popA, fitA))
    popB = reproduce(select(popB, fitB))

    print(f"Gen {gen}: Fitness A={fitA.mean():.3f}, Fitness B={fitB.mean():.3f}")

## Final Approximation

We can visualize how well the evolved solution approximates the target function.

In [ ]:
x = np.linspace(-5, 5, 200)

ys = []
gens = []

for gen, a, b in best_solutions:
    y = component_a(x, a) + component_b(x, b)
    ys.append(y)
    gens.append(gen)

ys = np.array(ys)
gens = np.array(gens)

In [ ]:
import matplotlib.cm as cm

fig, ax = plt.subplots()

norm = plt.Normalize(gens.min(), gens.max())
cmap = cm.viridis

for i in range(len(gens)):
    ax.plot(x, ys[i], color=cmap(norm(gens[i])), alpha=0.7)

# Plot target
ax.plot(x, target_function(x), color='black', linewidth=3, label="Target")

# Create scalar mappable
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

# Attach colorbar to the figure + axes
fig.colorbar(sm, ax=ax, label="Generation")

ax.set_title("Cooperative Coevolution Progress (Color = Generation)")
ax.legend()

plt.show()